# tnbc-92 · summary UMAPs across all 8 runs

interactive Plotly visualizations summarizing H1/H2/H3 results. one UMAP per alignment run, computed on the 45,661 held-out test niches' z_he embeddings, colored multiple ways to visualize the biology axes.

### scale clarification (read this first)

**every dot in every UMAP is one niche, NOT one H&E section.**

```
1 niche  =  1 center spot + 6 spatial neighbors  ~  1200 cells  (~100µm footprint x 7 spots)
1 H&E section (subarray)  ~  1000-1500 niches  ->  1000-1500 dots
14 held-out test patients  ->  38 H&E sections  ->  45,661 niches  ->  subsampled to 8,000 dots per panel
```

so when you see a connected cloud of ~1000 dots in the UMAP, that is roughly one H&E section. when you see colored gradients across the manifold, those are biology gradients across niches - physical regions of the tumor architecture.

**hover any dot** to see `niche_id`, `subarray` (one H&E section), `patient_id`, and the colored attribute value.

### dot identity across panels
- within a section, the 8 subplots (one per run) draw the SAME set of niches (same `seed=42`, same label-filter), so dots correspond cross-panel for direct visual comparison.
- across sections, the 8k subsample differs because each section filters on a different label (`mc_megacluster`, `archetype`, `compartment`, ...) before sampling. don't try to track an individual niche from §3 -> §7.
- colors are globally stable: the same label gets the same color in every panel of every section (precomputed `label_color_map`).

### HTML export note
`pio.renderers.default = 'notebook'` is set in cell 1, so `jupyter nbconvert --to html 05_summary_umaps.ipynb` produces a self-contained HTML where hover/zoom/legend-toggle survive. each output cell embeds the plotly.js bundle (~3MB per fig). prefer the shell `nbconvert` command over JupyterLab's "Export As -> HTML" menu, which sometimes strips the mimebundle.

### what each panel shows
- **§3 mc_megacluster** (Wang 14-class niche-level biology, H2 part A rescue label): R1/R6 show clearest biology clusters (H2 winners). R4 lands in a tight manifold (geometry bias on K-means, linear probe parity).
- **§4 archetype** (proposal-literal H2 part A, 9-class **patient-level pseudobulk**): the label the proposal names; verified patient-level so every niche in a patient inherits one archetype.
- **§5 construct-validity** (archetype vs patient_id paired for B1 and R4): the visual answer to "is the proposal H2 part A evaluation question good?" - if archetype panel and patient_id panel show the same structure, the metric is collapsed to patient classification.
- **§6 patient_id** (14 held-out patients): CCA (B1) shows tight per-patient clusters (anti-helpful). R4 mixes patients best (lowest patient z = 22.97). visualizes the patient-leakage axis.
- **§7 compartment** (proposal H2 part B, 17 tissue categories): per-spot histopathologist labels (Tumor, High/Low TIL stroma, TLS, Necrosis, ...) - spot-level, not patient-collapsed.
- **§8 three-space triptych** (z_he / z_st / z_mean) for R4 and B1: proposal H2's "three embedding spaces" within the shared latent, two color overlays (archetype, mc_megacluster).
- **§9 AUCell pathway scores**: continuous per-niche pathway activity (5 named Reactome parents). spatial gradients show where each pathway fires across the niche manifold.
- **§10 R4 DAG canonical directions**: R4 z_he projected onto specific sub-pathway directions (BTLA checkpoint, IFN-α/β regulation, integrin cell surface, ECM proteoglycans). visualizes Reactome DAG hierarchy finding that R4 encodes distinct sub-programs at DAG depth.

data sources: `runs/tnbc-92/{run}/embeddings_test.parquet`, `data/embeddings/biological_signals/niche_aucell_5targets.parquet`, `data/embeddings/niches/*.parquet`, `runs/tnbc-92/eval/H3/pathway_cca/dag_full/`.

UMAP caching: all UMAPs cached to `runs/tnbc-92/eval/figures/umap_cache/{run}_{view}_{hp}.parquet`. the `{hp}` suffix is a 6-char hash of (metric, n_neighbors, min_dist, seed) so bumping any hyperparam writes a fresh file rather than silently re-serving stale coords.

## 0 · methods note — gpath2vec input, AUCell, v1/v2 (read first)

**terminology.** `arm_A` is **v2** throughout (arms B/C were planned, never run). **v1** = `gpath2vec_output/full_cohort_tf_low/` (`_v1_legacy`, pre-F1, the build the R-runs ingested on ST; not regenerable). **v2** = `full_cohort_tf_low_arm_A/` (post-F1 min-FDR fix; the §13 H3 probe). Different builds.

**per-niche pathway signal (code-verified, `gpath2vec/cli.py niche-pipeline`):**

- **Fisher path (v1 and v2):** per niche, take the **top-100 highest-EXPRESSED genes** (`top = np.argsort(row)[-top_genes:]; top = top[row[top]>0]`, `--top-genes` default 100), Fisher's exact vs the Reactome universe, per-niche BH-FDR, min-FDR aggregation (F1 fix in v2). The `TF gene filter: 1658` in v2's log is a separate *pathway-universe* restriction (`filter_pathways(gene_filter=TF)`), **not** a per-niche gene selection.
- **AUCell path (new; the in-progress rerun, not §13):** rank *all* genes by niche pseudobulk (normalize_total 1e4 → log1p), integrate each gene-set recovery curve over the top ~5% (decoupler `dc.mt.aucell`). **No per-niche gene selection**; sparsity set later by `topk_per_niche` (top-k pathways/niche, default 50).

**there is no MAD/mean (variability) gene filter anywhere in gpath2vec** — package-wide verified. The only per-niche selection that exists is *top-N by expression*. So "v2 is not MAD/mean" is true, but not because a MAD/mean step was dropped — one was never implemented.

**what v2 (=arm_A, the §13 probe) input actually was:** top-100-expressed genes/niche → TF-restricted, **level=low** Reactome universe → Fisher EA → min-FDR → metapath2vec (dim 512, 5 epochs). **Hypothesis (untested):** top-*expression* selection favors high-abundance / housekeeping / library-size-driven genes over discriminative ones, and `level=low` narrows coverage — together a plausible reason §13's gpath2vec H3 is weak/narrow (sig only on well-covered Immune/ECM, fails PCD, beaten by AUCell 3/4). The principled fix is the AUCell→graph path at `level=all` (no per-niche expression selection) — the rerun in progress. This is a mechanism hypothesis, not a verified cause.

In [1]:
import warnings
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import umap
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# embed full plotly.js bundle in each output cell so HTML export preserves hover/zoom/legend
pio.renderers.default = 'notebook'

# repo root = nearest ancestor (including cwd) containing projects/tnbc-92
_HERE = Path().resolve()
ROOT = next((p for p in [_HERE, *_HERE.parents] if (p / 'projects' / 'tnbc-92').exists()), None)
assert ROOT is not None, f'could not locate repo root from {_HERE}'

RUNS = ROOT / 'runs' / 'tnbc-92'
BIO = ROOT / 'data' / 'embeddings' / 'biological_signals'
NICHES = ROOT / 'data' / 'embeddings' / 'niches'
CACHE = RUNS / 'eval' / 'figures' / 'umap_cache'
CACHE.mkdir(parents=True, exist_ok=True)

RUN_IDS = ['R1', 'R2', 'R3', 'R4', 'R6', 'B1', 'B2', 'B3']
PATHWAYS = ['TGF-beta_Signaling', 'Immune_System', 'ECM_Organization',
            'Cell_Cycle', 'Programmed_Cell_Death']
SEED = 42
SAMPLE_SIZE = 8000  # subsample for Plotly performance (full 45k is sluggish)

# global palette used for ALL discrete-label panels
PALETTE = (px.colors.qualitative.Set3 + px.colors.qualitative.Pastel +
           px.colors.qualitative.Bold)

print(f'repo root: {ROOT.name}')
print(f'plotly renderer: {pio.renderers.default}')

repo root: omicstra
plotly renderer: notebook


## 1 · load embeddings + biology labels + AUCell pathway scores

In [2]:
def _as_arr(v):
    return np.asarray(v, dtype=np.float32)

def stack_col(s):
    return np.stack([_as_arr(v) for v in s])


def load_run(run_id):
    df = pd.read_parquet(RUNS / run_id / 'embeddings_test.parquet').reset_index()
    df['niche_id'] = df['subarray'].astype(str) + '::' + df['spot_id'].astype(str)
    return df


# load AUCell 5-pathway scores (per-subarray z-scored columns)
aucell = pd.read_parquet(BIO / 'niche_aucell_5targets.parquet')
aucell_cols = ['niche_id'] + [f'{p}_z' for p in PATHWAYS]
aucell = aucell.reset_index()[aucell_cols]
print(f'aucell: {aucell.shape}')

# load mc_megacluster per niche
rows = []
for p in sorted(NICHES.glob('*.parquet')):
    df = pd.read_parquet(p, columns=['subarray', 'mc_megacluster'])
    df = df.reset_index() if df.index.name == 'spot_id' else df
    if 'spot_id' not in df.columns:
        continue
    df['niche_id'] = df['subarray'].astype(str) + '::' + df['spot_id'].astype(str)
    rows.append(df[['niche_id', 'mc_megacluster']])
niche_mc = pd.concat(rows, ignore_index=True)
print(f'niche_mc: {niche_mc.shape}')

# patient + spot metadata - take from R1's embeddings (all runs share the same test split)
# archetype and compartment are already columns in embeddings_test.parquet (proposal H2 labels)
r1 = load_run('R1')
meta = r1[['niche_id', 'subarray', 'spot_id', 'patient_id', 'archetype', 'compartment']].copy()
# normalize empty-string compartment -> NaN
meta['compartment'] = meta['compartment'].astype('string').replace({'': pd.NA})
meta = meta.merge(niche_mc, on='niche_id', how='left')
meta = meta.merge(aucell, on='niche_id', how='left')
print(f'meta with biology labels + AUCell: {meta.shape}')
print(f'  mc_megacluster non-null: {meta["mc_megacluster"].notna().sum():,}')
print(f'  archetype non-null:     {meta["archetype"].notna().sum():,}  (proposal H2 part A label, PATIENT-level)')
print(f'  compartment non-null:   {meta["compartment"].notna().sum():,}  (proposal H2 part B label, spot-level)')
print(f'  AUCell non-null:        {meta[PATHWAYS[0] + "_z"].notna().sum():,}')


def _to_label_strs(s, kind):
    """normalize a Series to string labels for both 'discrete' (int) and 'discrete-str' kinds.
    using a stable cast prevents the same label rendering as '1' vs '1.0' across panels."""
    if kind == 'discrete':
        return pd.to_numeric(s, errors='coerce').astype('Int64').astype(str)
    return s.astype(str)


def label_color_map(col, kind='discrete'):
    """precompute a stable {label -> color} dict from the FULL meta frame.
    used by every discrete panel so the same label gets the same color across all subplots
    and across all sections, regardless of per-subplot sampling."""
    labels = _to_label_strs(meta[col].dropna(), kind)
    uniq = sorted(pd.unique(labels))
    return {v: PALETTE[k % len(PALETTE)] for k, v in enumerate(uniq)}

aucell: (286250, 6)
niche_mc: (277695, 2)
meta with biology labels + AUCell: (45661, 12)
  mc_megacluster non-null: 44,192
  archetype non-null:     45,661  (proposal H2 part A label, PATIENT-level)
  compartment non-null:   15,953  (proposal H2 part B label, spot-level)
  AUCell non-null:        45,661


## 2 · compute (or load cached) UMAPs for each run

In [3]:
def _umap_param_hash(n_neighbors, min_dist, seed, metric='cosine'):
    """short fingerprint of UMAP hyperparams for cache keys.
    bumping any of these returns a different file rather than silently re-serving stale coords."""
    s = f'{metric}_n{n_neighbors}_md{min_dist}_s{seed}'
    return hashlib.md5(s.encode()).hexdigest()[:6]


def umap_for_run(run_id, view='z_he', n_neighbors=30, min_dist=0.1):
    """compute or load cached UMAP for one run's test embeddings."""
    h = _umap_param_hash(n_neighbors, min_dist, SEED)
    cache_path = CACHE / f'{run_id}_{view}_{h}.parquet'
    if cache_path.exists():
        return pd.read_parquet(cache_path)
    df = load_run(run_id)
    Z = stack_col(df[view])
    # L2-normalize so the UMAP cosine metric is consistent across runs
    Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    print(f'  {run_id} {view} (hp={h}): fitting UMAP (n={len(Z)}, d={Z.shape[1]})...', flush=True)
    reducer = umap.UMAP(
        n_neighbors=n_neighbors, min_dist=min_dist,
        metric='cosine', random_state=SEED, verbose=False,
    )
    coords = reducer.fit_transform(Z)
    out = pd.DataFrame({
        'niche_id': df['niche_id'].values,
        'umap_x': coords[:, 0],
        'umap_y': coords[:, 1],
    })
    out.to_parquet(cache_path)
    return out


umaps = {}
for r in RUN_IDS:
    umaps[r] = umap_for_run(r, 'z_he')
    h = _umap_param_hash(30, 0.1, SEED)
    print(f'    {r}: cached at {(CACHE / f"{r}_z_he_{h}.parquet").relative_to(ROOT)}')

    R1: cached at runs/tnbc-92/eval/figures/umap_cache/R1_z_he_e0128e.parquet
    R2: cached at runs/tnbc-92/eval/figures/umap_cache/R2_z_he_e0128e.parquet
    R3: cached at runs/tnbc-92/eval/figures/umap_cache/R3_z_he_e0128e.parquet
    R4: cached at runs/tnbc-92/eval/figures/umap_cache/R4_z_he_e0128e.parquet
    R6: cached at runs/tnbc-92/eval/figures/umap_cache/R6_z_he_e0128e.parquet
    B1: cached at runs/tnbc-92/eval/figures/umap_cache/B1_z_he_e0128e.parquet
    B2: cached at runs/tnbc-92/eval/figures/umap_cache/B2_z_he_e0128e.parquet
    B3: cached at runs/tnbc-92/eval/figures/umap_cache/B3_z_he_e0128e.parquet


## 3 · per-run UMAP colored by Wang mc_megacluster (H2 biology axis)

this visualizes H2 part A-MC: do niches cluster by Wang's per-spot 14-class biological identity? R1/R6 should show clearest clusters (winners on K-means MC ARI 0.23-0.25). R4 lives in a tight 3-effective-dim manifold (KMeans ARI 0.098 but linear probe parity).

In [4]:
def make_panel_figure(color_col, color_kind='discrete', cmap=None,
                      title='', sample_size=SAMPLE_SIZE, color_label=None):
    """2x4 grid (R1-R4 top; R6,B1,B2,B3 bottom), one run per subplot, UMAP
    colored by `color_col`.

    render-stable design: uses SVG go.Scatter (NOT Scattergl) so the figure
    survives notebook reloads/scroll - a 2x4 Scattergl grid spends 8 WebGL
    contexts per figure and the browser (~16 cap) evicts them across the
    notebook's many gl figures, which is why these panels "would not stay
    loaded". one data trace per subplot (per-point color array) instead of
    one-per-label; discrete legend via a single set of dummy traces. global
    label->color map keeps the SAME label the SAME color across all panels.

    color_kind: 'discrete' (int labels) | 'discrete-str' (str labels) |
    'continuous' (float + colorscale).
    """
    fig = make_subplots(rows=2, cols=4, subplot_titles=RUN_IDS,
                        horizontal_spacing=0.04, vertical_spacing=0.12)
    color_map = (label_color_map(color_col, color_kind)
                 if color_kind != 'continuous' else None)
    # SVG is slower per point than gl; cap per-panel points for snappy,
    # reload-stable render (8 panels x cap total).
    cap = min(sample_size, 4000)

    for i, run_id in enumerate(RUN_IDS):
        row, col = i // 4 + 1, i % 4 + 1
        df = umaps[run_id].merge(meta, on='niche_id', how='left')
        df = df.dropna(subset=['umap_x', 'umap_y', color_col])
        if len(df) > cap:
            df = df.sample(cap, random_state=SEED)
        hov = [f'1 niche (~1200 cells)<br>niche_id: {nid}<br>'
               f'subarray (H&E section): {sub}<br>patient_id: {pat}'
               for nid, sub, pat in zip(df['niche_id'], df['subarray'],
                                        df['patient_id'])]
        if color_kind in ('discrete', 'discrete-str'):
            vals = _to_label_strs(df[color_col], color_kind).values
            pt_cols = [color_map.get(v, '#cccccc') for v in vals]
            fig.add_trace(go.Scatter(
                x=df['umap_x'], y=df['umap_y'], mode='markers',
                marker=dict(size=3, color=pt_cols, opacity=0.7),
                customdata=vals, text=hov, showlegend=False,
                hovertemplate=(f'<b>{color_col}: %{{customdata}}</b><br>'
                               f'%{{text}}<extra></extra>')),
                row=row, col=col)
        else:
            fig.add_trace(go.Scatter(
                x=df['umap_x'], y=df['umap_y'], mode='markers',
                marker=dict(size=3, color=df[color_col].astype(float).values,
                            colorscale=cmap or 'Viridis', showscale=(i == 0),
                            colorbar=dict(title=color_label or color_col,
                                          x=1.02, len=0.4, y=0.78)
                            if i == 0 else None),
                text=hov, showlegend=False,
                hovertemplate=(f'<b>{color_col}: %{{marker.color:.3f}}</b>'
                               f'<br>%{{text}}<extra></extra>')),
                row=row, col=col)
        fig.update_xaxes(showticklabels=False, row=row, col=col)
        fig.update_yaxes(showticklabels=False, row=row, col=col)

    # discrete legend: one cheap zero-point dummy trace per label (added once)
    if color_map is not None:
        for v in color_map:
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode='markers',
                marker=dict(size=9, color=color_map[v]),
                name=str(v), legendgroup=str(v), showlegend=True),
                row=1, col=1)

    fig.update_layout(
        title=(f'{title}<br><sub>each dot = 1 niche (~1200 cells, 7-spot '
               f'footprint). 1 H&E section ~ 1000 dots. {cap:,} of 45,661 '
               f'test niches shown per panel (SVG, reload-stable).</sub>'),
        height=600, width=1200, margin=dict(l=20, r=120, t=110, b=20),
        legend=dict(itemsizing='constant', x=1.02, y=1.0))
    return fig


fig = make_panel_figure(
    'mc_megacluster', color_kind='discrete',
    title='UMAP of z_he · colored by Wang mc_megacluster (14-class niche-level biology)',
)
fig.show()

## 4 · per-run UMAP colored by Wang 9 spatial archetypes (proposal-literal H2 part A label)

this is the label the original proposal names in H2: *"the nine spatial archetypes will cluster more coherently in the aligned latent space"*. colored here so you can see what the label actually looks like across the niche manifold.

**caveat (verified in `projects/tnbc-92/proposal_deviations.md`):** Wang's `archetype` is sourced from `Spatial archetypes_defined_on_ST_global_pseudobulk` in Clinical.RDS - a **patient-level pseudobulk label**. 30/30 sampled subarrays have unique archetype values; every niche in a patient inherits one archetype. ARI on this label is mechanically a 9-class patient classification at niche resolution, not a spatial-coherence test. compare this panel side-by-side with §6 patient_id (same color structure ≈ collapsed metric). the niche-level biology rescue is `mc_megacluster` in §3.

In [5]:
fig = make_panel_figure(
    'archetype', color_kind='discrete',
    title='UMAP of z_he · colored by Wang 9 spatial archetypes (PATIENT-LEVEL pseudobulk label, proposal-literal H2 part A)',
)
fig.show()

## 5 · construct-validity · 9-archetype vs patient_id on the same UMAP (B1 vs R4)

if the 9-archetype panel and the patient_id panel show the same color structure for a given run, "9-archetype ARI" is mechanically a patient classification on that run. quantified via NMI(archetype, patient_id) on the 14 held-out test patients - archetype is a deterministic function of patient (every niche in a patient inherits one archetype) so NMI is high by construction.

B1 (CCA baseline) is the worst offender: it collapses to patient identity (patient z=122.13, 3.3x amplification over raw_he) and so its archetype clusters look identical to its patient clusters. R4 (cross-attention) mixes patients best (patient z=22.97) so its archetype-colored panel shows less per-patient blocking despite the label itself being patient-derived.

this is the visual answer to the evaluation-question concern - "is the eval question actually a good thing to ask in order to evaluate the objective?" - no, not as the proposal literally writes H2 part A, because the label is collapsed to patient identity. that's the construct-validity issue the proposal H2 walked into and the deviations log corrects.

In [6]:
from sklearn.metrics import normalized_mutual_info_score

# verify archetype is a function of patient on the held-out test cohort
test_meta = meta.dropna(subset=['archetype', 'patient_id']).copy()
patient_arch = test_meta.groupby('patient_id')['archetype'].nunique()
n_test_patients = len(patient_arch)
n_unique = (patient_arch == 1).sum()
nmi_all = normalized_mutual_info_score(
    test_meta['patient_id'].astype(str).values,
    _to_label_strs(test_meta['archetype'], 'discrete').values,
)
print(f'test cohort: {n_test_patients} patients, {n_unique}/{n_test_patients} with unique archetype across all their niches')
print(f'NMI(archetype, patient_id) on test niches: {nmi_all:.3f}  (1.0 = archetype = f(patient))')


# 2x2 paired figure: rows = [B1, R4], cols = [archetype overlay, patient_id overlay]
def paired_construct_validity(run_ids=('B1', 'R4'), sample_size=SAMPLE_SIZE):
    titles = []
    for r in run_ids:
        titles += [f'{r} · 9 spatial archetypes (proposal label)',
                   f'{r} · patient_id (14 held-out)']
    fig = make_subplots(rows=len(run_ids), cols=2, subplot_titles=titles,
                        horizontal_spacing=0.05, vertical_spacing=0.15)
    # global color maps so colors are stable across both rows and both columns
    cmap_arch = label_color_map('archetype', 'discrete')
    cmap_pat = label_color_map('patient_id', 'discrete-str')
    col_maps = {'archetype': ('discrete', cmap_arch),
                'patient_id': ('discrete-str', cmap_pat)}
    for r_idx, run_id in enumerate(run_ids):
        df = umaps[run_id].merge(meta, on='niche_id', how='left')
        df = df.dropna(subset=['umap_x', 'umap_y', 'archetype', 'patient_id'])
        if len(df) > sample_size:
            df = df.sample(sample_size, random_state=SEED)
        for c_idx, color_col in enumerate(['archetype', 'patient_id']):
            kind, cmap = col_maps[color_col]
            vals = _to_label_strs(df[color_col], kind).values
            for v in cmap.keys():
                mask = vals == v
                if not mask.any():
                    continue
                hover_text = [
                    f'1 niche (~1200 cells)<br>niche_id: {nid}<br>subarray: {sub}<br>patient_id: {pat}<br>archetype: {a}'
                    for nid, sub, pat, a in zip(df.loc[mask, 'niche_id'],
                                                df.loc[mask, 'subarray'],
                                                df.loc[mask, 'patient_id'],
                                                df.loc[mask, 'archetype'])
                ]
                fig.add_trace(
                    go.Scattergl(
                        x=df.loc[mask, 'umap_x'], y=df.loc[mask, 'umap_y'],
                        mode='markers',
                        marker=dict(size=3, color=cmap[v], opacity=0.7),
                        name=f'{color_col}={v}',
                        showlegend=False, text=hover_text,
                        hovertemplate='%{text}<extra></extra>',
                    ),
                    row=r_idx + 1, col=c_idx + 1)
            fig.update_xaxes(showticklabels=False, row=r_idx + 1, col=c_idx + 1)
            fig.update_yaxes(showticklabels=False, row=r_idx + 1, col=c_idx + 1)
    fig.update_layout(
        title=(f'construct-validity · same UMAP, two color overlays.<br>'
               f'<sub>if the archetype panel and patient_id panel show the same structure, '
               f'"9-archetype ARI" is mechanically a patient classification on that run. '
               f'NMI(archetype, patient_id) on test cohort = {nmi_all:.3f}.</sub>'),
        height=700, width=1100,
        margin=dict(l=20, r=40, t=110, b=20),
    )
    return fig


fig = paired_construct_validity(('B1', 'R4'))
fig.show()

test cohort: 14 patients, 14/14 with unique archetype across all their niches
NMI(archetype, patient_id) on test niches: 0.897  (1.0 = archetype = f(patient))


## 6 · per-run UMAP colored by patient_id (patient-leakage diagnostic)

shows H3-extended patient-axis preservation visually. B1 (CCA) should show tight per-patient clusters (patient z = 122.13, anti-helpful, 3.3x amplification over raw_he). R4 should mix patients best (patient z = 22.97, lowest in grid).

read this panel together with §4 (archetype) and §5 (construct-validity): if a run's archetype panel and its patient_id panel are structurally identical, "9-archetype ARI" on that run is mechanically a patient classification.

In [7]:
fig = make_panel_figure(
    'patient_id', color_kind='discrete',
    title='UMAP of z_he · colored by patient_id (14 held-out patients) - visualizes patient leakage',
)
fig.show()

## 7 · per-run UMAP colored by tissue compartments (proposal H2 part B label)

proposal H2 part B specifies "per-compartment mean cosine similarity across 17 tissue categories" - this is the per-spot histopathologist-annotated compartment label (Tumor, High/Low TIL stroma, Necrosis, Lymphocyte, Stroma cell, Fat tissue, Vessels, ...). these labels DO vary at niche resolution (unlike the archetype label in §4), so they support a real spatial-coherence test of "where morphology and expression reinforce versus diverge".

**coverage caveat:** compartment annotations cover ~35% of the held-out test niches (15,953 / 45,661); 12 distinct classes populated in the test cohort, 15 in the full cohort (proposal cited 17 from Wang's annotation set). this panel filters to niches with non-null compartment. test-cohort class breakdown is printed below for transparency.

In [8]:
# coverage check on the test cohort
n_with_comp = meta['compartment'].notna().sum()
n_classes = meta['compartment'].dropna().nunique()
print(f'test niches with non-null compartment: {n_with_comp:,} / {len(meta):,}  ({100*n_with_comp/len(meta):.1f}% covered)')
print(f'compartment classes present in test cohort: {n_classes}')
print('compartment class counts:')
print(meta['compartment'].value_counts().to_string())

fig = make_panel_figure(
    'compartment', color_kind='discrete-str',
    title='UMAP of z_he · colored by tissue compartments (proposal H2 part B label · spot-level)',
)
fig.show()

test niches with non-null compartment: 15,953 / 45,661  (34.9% covered)
compartment classes present in test cohort: 12
compartment class counts:
compartment
Acellular stroma    4670
Tumor               4384
Lymphocyte          2128
Stroma cell         2098
Necrosis            1178
Fat tissue           887
Low TIL stroma       354
High TIL stroma      105
Lactiferous duct      45
Lymphoid nodule       44
Vessels               42
in situ               18


## 8 · three-space triptych (z_he / z_st / z_mean) for R4 and B1 — proposal H2 across 3 embedding spaces

proposal H2 calls for evaluating "across three embedding spaces: H&E-only, ST-only, shared aligned". this panel shows three views within the shared 512-d latent for the two interesting runs:
- **R4 (cross-attention)**: H1 retrieval winner, H3 cross-patient winner, geometry-biased on K-means MC ARI
- **B1 (CCA classical baseline)**: patient-collapsed, useful as construct-validity foil

each row is one run, three columns = (z_he projected H&E view, z_st projected ST view, z_mean joint view = (z_he+z_st)/2 L2-normed). colored by the proposal-literal `archetype` and by the niche-level `mc_megacluster`, side by side.

**proposal-literal caveat:** "H&E-only" and "ST-only" in the original proposal strictly mean the **un-aligned** modality embeddings (raw Virchow2 1280-d niche aggregate, raw Novae 64-d niche). these three triptych columns are the **post-projection** views, all living in the shared 512-d space. for the proposal-strict comparison run the H2 ARI eval on `data/embeddings/niches/*.parquet` columns `virchow2_niche` and `novae_niche` directly — out of scope for this UMAP notebook.

In [9]:
# compute z_st and z_mean UMAPs for R4 and B1 (z_he already cached from §2)
def umap_for_run_view(run_id, view, n_neighbors=30, min_dist=0.1):
    """cache-aware UMAP; supports view='z_mean' = (z_he+z_st)/2 L2-normed.
    cache key includes hyperparam hash so bumping n_neighbors/min_dist returns a new file."""
    h = _umap_param_hash(n_neighbors, min_dist, SEED)
    cache_path = CACHE / f'{run_id}_{view}_{h}.parquet'
    if cache_path.exists():
        return pd.read_parquet(cache_path)
    df = load_run(run_id)
    if view == 'z_mean':
        Z = (stack_col(df['z_he']) + stack_col(df['z_st'])) / 2.0
    else:
        Z = stack_col(df[view])
    Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    print(f'  {run_id} {view} (hp={h}): fitting UMAP (n={len(Z)}, d={Z.shape[1]})...', flush=True)
    reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, metric='cosine',
                        random_state=SEED, verbose=False)
    coords = reducer.fit_transform(Z)
    out = pd.DataFrame({'niche_id': df['niche_id'].values,
                        'umap_x': coords[:, 0], 'umap_y': coords[:, 1]})
    out.to_parquet(cache_path)
    return out


TRIPTYCH_VIEWS = ['z_he', 'z_st', 'z_mean']
TRIPTYCH_RUNS = ['R4', 'B1']
view_umaps = {}
for r in TRIPTYCH_RUNS:
    for v in TRIPTYCH_VIEWS:
        view_umaps[(r, v)] = umap_for_run_view(r, v)
        print(f'    {r} {v}: ready')


def triptych_panel(color_col, kind='discrete-str', sample_size=SAMPLE_SIZE):
    titles = [f'{r} · {v}' for r in TRIPTYCH_RUNS for v in TRIPTYCH_VIEWS]
    fig = make_subplots(rows=len(TRIPTYCH_RUNS), cols=len(TRIPTYCH_VIEWS),
                        subplot_titles=titles,
                        horizontal_spacing=0.04, vertical_spacing=0.15)
    # global color map so the same label paints the same hue in every one of the 6 panels
    cmap = label_color_map(color_col, kind)
    seen_legend = set()
    for r_idx, run_id in enumerate(TRIPTYCH_RUNS):
        for c_idx, view in enumerate(TRIPTYCH_VIEWS):
            df = view_umaps[(run_id, view)].merge(meta, on='niche_id', how='left')
            df = df.dropna(subset=['umap_x', 'umap_y', color_col])
            if len(df) > sample_size:
                df = df.sample(sample_size, random_state=SEED)
            vals = _to_label_strs(df[color_col], kind).values
            for v in cmap.keys():
                mask = vals == v
                if not mask.any():
                    continue
                hover_text = [
                    f'1 niche (~1200 cells)<br>niche_id: {nid}<br>subarray: {sub}<br>patient_id: {pat}'
                    for nid, sub, pat in zip(df.loc[mask, 'niche_id'],
                                             df.loc[mask, 'subarray'],
                                             df.loc[mask, 'patient_id'])
                ]
                show_legend = v not in seen_legend
                seen_legend.add(v)
                fig.add_trace(
                    go.Scattergl(
                        x=df.loc[mask, 'umap_x'], y=df.loc[mask, 'umap_y'],
                        mode='markers',
                        marker=dict(size=3, color=cmap[v], opacity=0.7),
                        name=str(v), legendgroup=str(v),
                        showlegend=show_legend, text=hover_text,
                        hovertemplate=f'<b>{color_col}: {v}</b><br>%{{text}}<extra></extra>',
                    ),
                    row=r_idx + 1, col=c_idx + 1)
            fig.update_xaxes(showticklabels=False, row=r_idx + 1, col=c_idx + 1)
            fig.update_yaxes(showticklabels=False, row=r_idx + 1, col=c_idx + 1)
    fig.update_layout(
        title=(f'three-space triptych · rows = run, cols = (z_he, z_st, z_mean=(z_he+z_st)/2). '
               f'colored by {color_col}.<br>'
               f'<sub>proposal H2 references three embedding spaces; this triptych shows the '
               f'three views within the shared 512-d latent. un-aligned modality view (raw '
               f'Virchow2, raw Novae) requires a separate eval and is deferred.</sub>'),
        height=700, width=1300,
        margin=dict(l=20, r=140, t=110, b=20),
        legend=dict(itemsizing='constant', x=1.02, y=1.0),
    )
    return fig

    R4 z_he: ready
    R4 z_st: ready
    R4 z_mean: ready
    B1 z_he: ready
    B1 z_st: ready
    B1 z_mean: ready


In [10]:
# kind='discrete' for archetype matches §4's int cast (avoids '1.0' vs '1' divergence)
fig_arch = triptych_panel('archetype', kind='discrete')
fig_arch.show()
fig_mc = triptych_panel('mc_megacluster', kind='discrete')
fig_mc.show()

### what is AUCell? (the pathway-activity score used in §9-§13)

**AUCell** = "Area Under the Curve" gene-set scoring (introduced in SCENIC, Aibar et al., *Nat. Methods* 2017; computed here via `decoupler.mt.aucell`). It answers one question per (niche, pathway): *are this pathway's genes among the most-expressed genes in this niche?*

**How it works — the recovery curve:**

1. Take one niche's expression vector. A *niche* here is a **pseudobulk** of the center spot + 6 spatial neighbours (~1200 cells); counts are `normalize_total`→1e4 then `log1p`.
2. **Rank all genes** in that niche from highest to lowest expression.
3. For a pathway's gene set, walk down the ranking and trace the **recovery curve**: x = how far down the ranking you have gone, y = how many of the pathway's genes have appeared so far.
4. **AUCell score = area under that recovery curve**, integrated over only the **top ~5%** of ranked genes (decoupler default), rescaled to ~[0, 1].

**Intuition:** if a pathway's genes pile up near the top of the niche's expression ranking, the curve rises steeply early → large area → score near 1 → **pathway active** in that niche. If the pathway's genes are scattered or lowly expressed, the curve crawls → small area → score near 0 → pathway inactive.

**Why we use it (properties that matter):**

- **Rank-based, not magnitude-based** → invariant to per-niche library size / scaling; only the *order* of genes within a niche matters, so scores are comparable across niches with no control group and no batch correction.
- **Per-niche and independent** → each niche scored on its own ranking; output is one continuous activity value per (niche, pathway), not a differential/contrast test.
- **No arbitrary per-niche gene cutoff** → it uses the *full* gene ranking (the top-5% window is on the ranking, not a hand-picked gene list). This is the key contrast with the Fisher / gpath2vec path, which must first select each niche's top-100 expressed genes; AUCell removes that choice (see §0).

**In this notebook:** AUCell scores the 5 Reactome target pathways (§9) and the 395 Reactome DAG sub-pathways (§10-§12) per niche, then per-subarray z-scored for display; it is also the member-matched comparator arm in the §13 gpath2vec head-to-head. "Activity" here means transcriptional activity of the pathway's gene programme in that ~1200-cell niche — **not** single-cell resolution.

## 9 · R4 UMAP colored by AUCell pathway scores (5 named Reactome parents)

the H3 cross-patient generalization test ran univariate CCA between R4's z_he canonical direction and each pathway's AUCell score. R4 wins 15/15 cells at z=3.8-5.7. these UMAPs show the spatial gradients: where each pathway fires across R4's manifold.

In [11]:
def make_pathway_panel(run_id, sample_size=SAMPLE_SIZE):
    """5 panels: same UMAP (same dots = same niches), each colored by a different pathway AUCell score.
    every dot = 1 niche (~1200 cells); 1 H&E section ~ 1000 dots."""
    df = umaps[run_id].merge(meta, on='niche_id', how='left')
    df = df.dropna(subset=['umap_x', 'umap_y'])
    if len(df) > sample_size:
        df = df.sample(sample_size, random_state=SEED)

    short = {p: p.replace('_Signaling','').replace('_Organization','').replace('_System','')
                 .replace('_Cycle','').replace('Programmed_Cell_Death','PCD').replace('TGF-beta','TGF-b')
             for p in PATHWAYS}
    fig = make_subplots(rows=1, cols=5, subplot_titles=[short[p] for p in PATHWAYS],
                        horizontal_spacing=0.03)
    hover_text = [
        f'1 niche (~1200 cells)<br>niche_id: {nid}<br>subarray (H&E section): {sub}<br>patient_id: {pat}'
        for nid, sub, pat in zip(df['niche_id'], df['subarray'], df['patient_id'])
    ]
    for i, p in enumerate(PATHWAYS):
        zcol = f'{p}_z'
        if zcol not in df.columns:
            continue
        fig.add_trace(
            go.Scattergl(
                x=df['umap_x'], y=df['umap_y'], mode='markers',
                marker=dict(size=3, color=df[zcol], colorscale='RdBu_r',
                            cmid=0, showscale=(i == 4),
                            colorbar=dict(title='AUCell z', x=1.02, len=0.9) if i == 4 else None),
                text=hover_text,
                hovertemplate=f'<b>{short[p]} AUCell z: %{{marker.color:.2f}}</b><br>%{{text}}<extra></extra>',
                showlegend=False,
            ),
            row=1, col=i + 1)
        fig.update_xaxes(showticklabels=False, row=1, col=i + 1)
        fig.update_yaxes(showticklabels=False, row=1, col=i + 1)

    fig.update_layout(
        title=(f'UMAP of z_he ({run_id}) · 5 named Reactome pathway AUCell scores (per-subarray z-scored)<br>'
               f'<sub>same {len(df):,} niches in all 5 panels; only the dot color changes between panels. '
               f'each dot = 1 niche (~1200 cells); 1 H&E section ~ 1000 dots.</sub>'),
        height=400, width=1400,
        margin=dict(l=20, r=100, t=110, b=20),
    )
    return fig


fig = make_pathway_panel('R4')
fig.show()

## 10 · R4 UMAP colored by top DAG canonical-direction projections (pathway hierarchy)

R4's z_he projected onto specific DAG-node canonical directions from `runs/tnbc-92/eval/H3/pathway_cca/dag_full/`. these are the named clinically interpretable sub-programs that R4 distinguishes at DAG resolution: BTLA co-inhibition (z=6.92), IL receptor SHC signaling (6.80), integrin cell surface interactions (6.21), ECM proteoglycans (5.73), IFN-α/β regulation (5.71). visualizing this confirms that R4's manifold encodes biologically distinct axes at sub-pathway resolution, refuting the original 5-parent aliasing reading.

In [12]:
# load DAG-level data: per-niche AUCell at every DAG node, plus the top R4 hits from per_node_cca
dag_dir = RUNS / 'eval' / 'H3' / 'pathway_cca' / 'dag_full'
dag_meta = pd.read_parquet(dag_dir / 'dag_metadata.parquet')
dag_cca = pd.read_parquet(dag_dir / 'per_node_cca.parquet')
dag_aucell = pd.read_parquet(dag_dir / 'per_node_aucell.parquet')

# pick R4's top-8 hits on z_he by option-A z-score; tie-break by node_id for stable ordering
r4_top = dag_cca[(dag_cca.run_id == 'R4') & (dag_cca.view == 'z_he')] \
    .sort_values(['z_A', 'node_id'], ascending=[False, True]).head(8).copy()
r4_top = r4_top.merge(dag_meta[['node_id', 'node_name', 'parent_name', 'depth_under_parent']],
                       on='node_id', how='left')
print('R4 top-8 DAG canonical-direction hits (z_he):')
print(r4_top[['node_id', 'node_name', 'parent_name', 'obs_A', 'z_A']].to_string(index=False))

# merge per-node AUCell scores into R4 UMAP
top_nodes = r4_top['node_id'].tolist()
r4_umap = umaps['R4'].merge(meta[['niche_id', 'subarray', 'patient_id']], on='niche_id', how='left')
au_sub = dag_aucell[['niche_id'] + top_nodes]
r4_umap = r4_umap.merge(au_sub, on='niche_id', how='left')
# per-subarray z-score the AUCell columns for visual consistency
for n in top_nodes:
    r4_umap[n] = r4_umap.groupby('subarray')[n].transform(
        lambda s: (s - s.mean()) / (s.std() + 1e-9))
print(f'\nmerged R4 UMAP with {len(top_nodes)} top DAG-node AUCell columns')

R4 top-8 DAG canonical-direction hits (z_he):
      node_id                                node_name      parent_name    obs_A      z_A
R-HSA-9927353                    Co-inhibition by BTLA    Immune_System 0.308275 6.923356
 R-HSA-912526       Interleukin receptor SHC signaling    Immune_System 0.307390 6.800148
 R-HSA-216083       Integrin cell surface interactions ECM_Organization 0.587410 6.206736
R-HSA-3000178                        ECM proteoglycans ECM_Organization 0.626904 5.731470
 R-HSA-912694        Regulation of IFNA/IFNB signaling    Immune_System 0.345119 5.714918
R-HSA-1474244        Extracellular matrix organization ECM_Organization 0.519109 5.486996
R-HSA-1474228  Degradation of the extracellular matrix ECM_Organization 0.507680 5.468338
 R-HSA-202433 Generation of second messenger molecules    Immune_System 0.528730 5.451586

merged R4 UMAP with 8 top DAG-node AUCell columns


In [13]:
def make_dag_panel(umap_df, r4_top, sample_size=SAMPLE_SIZE):
    """8 panels: same R4 UMAP (same dots = same niches), each colored by a different DAG-node AUCell score.
    every dot = 1 niche (~1200 cells); 1 H&E section ~ 1000 dots. visualizes pathway hierarchy finding that
    R4 encodes distinct sub-pathway programs at DAG depth."""
    df = umap_df.dropna(subset=['umap_x', 'umap_y']).copy()
    if len(df) > sample_size:
        df = df.sample(sample_size, random_state=SEED)

    titles = []
    for _, r in r4_top.iterrows():
        nm = r['node_name'][:32] + ('...' if len(r['node_name']) > 32 else '')
        titles.append(f"{nm}<br><sub>{r['parent_name']} · z={r['z_A']:.2f}</sub>")

    fig = make_subplots(rows=2, cols=4, subplot_titles=titles,
                        horizontal_spacing=0.04, vertical_spacing=0.18)
    for i, (_, r) in enumerate(r4_top.iterrows()):
        row = i // 4 + 1
        col = i % 4 + 1
        node_id = r['node_id']
        if node_id not in df.columns:
            continue
        sub = df.dropna(subset=[node_id])
        hover_text = [
            f'1 niche (~1200 cells)<br>niche_id: {nid}<br>subarray (H&E section): {s}<br>patient_id: {pat}'
            for nid, s, pat in zip(sub['niche_id'], sub['subarray'], sub['patient_id'])
        ]
        fig.add_trace(
            go.Scattergl(
                x=sub['umap_x'], y=sub['umap_y'], mode='markers',
                marker=dict(size=3, color=sub[node_id], colorscale='RdBu_r',
                            cmid=0, cmin=-3, cmax=3,
                            showscale=(i == 3),
                            colorbar=dict(title='AUCell z<br>(per-sub)', x=1.02, len=0.4,
                                          y=0.78) if i == 3 else None),
                text=hover_text,
                hovertemplate=f'<b>{r["node_name"]}: %{{marker.color:.2f}}</b><br>%{{text}}<extra></extra>',
                showlegend=False,
            ),
            row=row, col=col)
        fig.update_xaxes(showticklabels=False, row=row, col=col)
        fig.update_yaxes(showticklabels=False, row=row, col=col)

    fig.update_layout(
        title=('R4 UMAP · top 8 DAG-node AUCell scores (pathway hierarchy, sorted by option-A z)<br>'
               '<sub>same R4 UMAP in all 8 panels; only the colored DAG-node AUCell score changes. '
               'each dot = 1 niche (~1200 cells); 1 H&E section ~ 1000 dots.</sub>'),
        height=700, width=1300,
        margin=dict(l=20, r=120, t=130, b=20),
    )
    return fig


fig = make_dag_panel(r4_umap, r4_top)
fig.show()

## 11 · H3 cross-patient generalization summary (DAG-resolution bar chart)

the headline H3 result from pathway hierarchy: percentage of DAG-tests (node × view) significant under BH-FDR<0.05 in the cross-patient option-A test (**1185 tests per run** = 395 DAG nodes × 3 views; 9480 is the total across all 8 runs, not per run; there is no cell-type dimension). R4 78.8%, B2 74.2%, B1 70.2%, B3 64.4%, R6 36.9%, R1 27.1%, R2 23.0%, R3 18.4% — classical baselines (B2/B1) are **competitive at DAG resolution** (commit-3.6: classical catch up at fine pathway resolution); late-fusion contrastive (R1/R2/R3) is lowest.

In [14]:
sig_summary = dag_cca.groupby('run_id').agg(
    n_tests=('node_id', 'size'),  # one row = one DAG-test (node x view x cell-type), NOT biological cells
    n_sig=('sig_A_two_05', 'sum'),
).reset_index()
sig_summary['pct_sig'] = 100 * sig_summary['n_sig'] / sig_summary['n_tests']
sig_summary = sig_summary.sort_values('pct_sig', ascending=False)

fam_color = {r: ('#1f77b4' if r.startswith('R') else '#ff7f0e') for r in RUN_IDS}

fig = go.Figure(go.Bar(
    x=sig_summary['run_id'], y=sig_summary['pct_sig'],
    marker_color=[fam_color[r] for r in sig_summary['run_id']],
    text=[f'{p:.1f}%<br>{n:,}/{t:,}' for p, n, t in zip(
        sig_summary['pct_sig'], sig_summary['n_sig'], sig_summary['n_tests'])],
    textposition='outside',
    hovertemplate='%{x}<br>sig: %{y:.1f}%<extra></extra>',
))
fig.add_hline(y=5, line_dash='dash', line_color='gray',
              annotation_text='5% FDR floor', annotation_position='right')
fig.update_layout(
    title='H3 cross-patient generalization · % of DAG-tests (node × view × cell-type) with BH-FDR < 0.05 per run',
    yaxis_title='% significant tests',
    xaxis_title='run',
    height=400, width=900,
    showlegend=False,
)
fig.show()

## 12 · R4 percent-sig per parent (DAG decomposition headline)

R4 wins or ties on every named parent. ECM is universally well-detected (small subtree, structural genes). Immune is moderately easy. Cell Cycle is hardest. visualizes how the named pathway parents differ in how easily H&E-only methods predict their AUCell-level activity cross-patient.

In [15]:
by_parent = dag_cca.merge(dag_meta[['node_id', 'parent_name']], on='node_id', how='left')
pct_per = by_parent.groupby(['run_id', 'parent_name'])['sig_A_two_05'].mean().unstack() * 100
# row order = current overall rank from sig_summary (auto-updates if rerun changes the ranking)
ordered_runs = sig_summary['run_id'].tolist()
pct_per = pct_per.reindex(ordered_runs)
pct_per = pct_per[['TGF-beta_Signaling', 'Immune_System', 'ECM_Organization',
                   'Cell_Cycle', 'Programmed_Cell_Death']]

fig = px.imshow(
    pct_per.values,
    x=['TGF-β', 'Immune', 'ECM', 'Cell Cycle', 'PCD'],
    y=pct_per.index.tolist(),
    color_continuous_scale='RdYlGn',
    aspect='auto',
    text_auto='.0f',
    labels=dict(color='% sig'),
)
fig.update_layout(
    title='H3 DAG decomposition · % significant DAG-tests per (run, parent), cross-patient option A',
    height=400, width=800,
    xaxis_title='Reactome parent', yaxis_title='run (ranked by overall % sig)',
)
fig.update_traces(textfont_size=14)
fig.show()

## 13 · H3 proposal-literal · gpath2vec pathway-CCA (arm A ≡ v2)

proposal H3 (verbatim): *"Reactome pathway embeddings computed via gpath2vec for five cancer-relevant pathways (TGF-β Signaling, Immune System, Extracellular Matrix Organization, Cell Cycle, Programmed Cell Death) will correlate with specific directions in the shared latent space via canonical correlation analysis, indicating preservation of interpretable biological signal."*

§9-§12 use **AUCell** as the per-niche pathway signal (the documented substitution). this section runs the **proposal-literal method**: gpath2vec pathway embeddings × shared latent via CCA. it is the first actual test of H3 as written, not a refinement of the AUCell readout. a null is a live, proposal-anticipated outcome ("negative or partial results establish the noise floor"); the direction is not pre-committed.

**provenance (locked).** embeddings = `data/embeddings/biological_signals/gpath2vec_output/full_cohort_tf_low_arm_A/embeddings.pkl`, sha256 `5705df87565db27dee1cefdb72583cf63c36d0c1e085daa2defb2421fbd7f8dc` (asserted in code). `*_v1_legacy*` is the irreproducible first-FDR artifact and is refused by assertion; its retired numbers (rho 0.9548, MC z 5.67, TIME z 6.52) are not cited.

**operationalization (not five clean vectors).** arm A is low-level (SBGN). the 5 named pathways are high-level (EHLD); none is a single graph node. each target's embedded low-level descendant pathways are tested **individually** (the CCA is per-member multivariate over the member set); the 5 named targets are a **grouping/parent label only - never a mean rollup vector**. two embedding objects are kept separate: the per-niche gpath2vec vector and each low-level pathway-node vector. no parent vector is formed; no averaging. coverage is uneven and shown per pathway.

**TGF-β is pinned, not chosen.** `R-HSA-170834` "Signaling by TGF-beta Receptor Complex" (per `proposal_deviations.md`), not the family-level `R-HSA-9006936`. member set = parent + 1 embedded descendant = **2 vectors**: underpowered by construction, reported for completeness, **excluded from any headline claim**.

**circularity quarantine.** R1/R2/R3/R4 consume gpath2vec on the ST input side, so `z_st`/`z_mean` for those runs are partially circular. only `z_he` is clean across all 8 runs. every per-run-view row carries `view_clean`; the headline uses **R4, z_he** only.

**reporting discipline.** no aggregate "X/5 significant" claim anywhere. per pathway with `set_size` and `coverage_frac` on every line. Immune (74) and ECM (14) are the only fairly-powered tests: a null there is a real **noise-floor** result, not "gpath2vec fails." Cell Cycle (~12% subtree), PCD (~18%), TGF-β (n=2) are **not fairly tested by construction**, never a method verdict.

source parquet: `runs/tnbc-92/eval/H3/pathway_cca_gpath2vec_vs_aucell_tflow/per_pathway_cca.parquet`, filtered `method=='gpath2vec'`. the `aucell_lowmatched` rows in that file are a separate in-flight controlled head-to-head and are not shown or duplicated here.

In [16]:
import pickle
from collections import defaultdict

# --- arm A provenance lock -------------------------------------------------
ARM_A = (ROOT / 'data' / 'embeddings' / 'biological_signals'
         / 'gpath2vec_output' / 'full_cohort_tf_low_arm_A' / 'embeddings.pkl')
EXPECTED_SHA = '5705df87565db27dee1cefdb72583cf63c36d0c1e085daa2defb2421fbd7f8dc'
assert 'legacy' not in ARM_A.name, f'refusing legacy embeddings: {ARM_A.name}'
_h = hashlib.sha256()
with open(ARM_A, 'rb') as _f:
    for _c in iter(lambda: _f.read(1 << 20), b''):
        _h.update(_c)
ARM_A_SHA = _h.hexdigest()
assert ARM_A_SHA == EXPECTED_SHA, f'arm A sha mismatch: {ARM_A_SHA}'
print(f'arm A embeddings.pkl sha256 verified: {ARM_A_SHA}')

# named targets + pinned ids (program.md lock; TGF-b = receptor-complex
# R-HSA-170834, NOT family-level R-HSA-9006936 - see proposal_deviations.md)
NAMED_TARGETS = {
    'TGF-beta_Signaling':    'R-HSA-170834',
    'Immune_System':         'R-HSA-168256',
    'ECM_Organization':      'R-HSA-1474244',
    'Cell_Cycle':            'R-HSA-1640170',
    'Programmed_Cell_Death': 'R-HSA-5357801',
}
FROZEN_SET_SIZES = {'TGF-beta_Signaling': 2, 'Immune_System': 74,
                    'ECM_Organization': 14, 'Cell_Cycle': 14,
                    'Programmed_Cell_Death': 7}
MIN_TESTABLE = 5  # shared >= 5 = fairly powered; below = not a method verdict

# --- canonical coverage denominator: PINNED to dag_metadata ---------------
# the AUCell-comparable, frozen Reactome hierarchy per proposal_deviations.md
# (dag_metadata.parquet INTERSECT per_node_cca.node_id = the frozen 395 nodes).
# this is the load-bearing pin: coverage_frac MUST use this, never a raw
# ReactomePathwaysRelation recompute (raw returns all-level descendants and
# drifts - the unpinned-denominator bug). raw recompute kept below as a
# printed transparency cross-check ONLY, not used in the table or prose.
_DAG = RUNS / 'eval' / 'H3' / 'pathway_cca' / 'dag_full'
_dm = pd.read_parquet(_DAG / 'dag_metadata.parquet')
_pnc = pd.read_parquet(_DAG / 'per_node_cca.parquet', columns=['node_id'])
_frozen = set(_dm['node_id']) & set(_pnc['node_id'])
CANON_TOTAL = (_dm[_dm['node_id'].isin(_frozen)]
               .groupby('parent_name')['node_id'].nunique().to_dict())
# hard pin: drift from the deviations-log figures fails loudly
_PINNED = {'Immune_System': 210, 'ECM_Organization': 20, 'Cell_Cycle': 120,
           'Programmed_Cell_Death': 38, 'TGF-beta_Signaling': 7}
assert CANON_TOTAL == _PINNED, (
    f'canonical denominator drift: dag_metadata frozen counts '
    f'{CANON_TOTAL} != deviations-log pin {_PINNED}. do not ship until '
    f'reconciled (this is the unpinned-denominator bug class).')
print(f'canonical coverage denominator pinned (dag_metadata frozen 395): '
      f'{CANON_TOTAL}')

# --- raw recompute (TRANSPARENCY CROSS-CHECK ONLY, never the table value) --
_ch = defaultdict(set)
with open(ROOT / 'knowledge' / 'reactome'
          / 'ReactomePathwaysRelation.txt') as f:
    for ln in f:
        p = ln.strip().split('\t')
        if len(p) == 2 and p[0].startswith('R-HSA-'):
            _ch[p[0]].add(p[1])


def _desc(root):
    seen, st = set(), [root]
    while st:
        n = st.pop()
        for c in _ch.get(n, ()):
            if c not in seen:
                seen.add(c)
                st.append(c)
    return seen


# --- load arm A; build member sets + per-member vectors --------------------
with open(ARM_A, 'rb') as f:
    _emb = pickle.load(f)
embedded = {k for k in _emb if isinstance(k, str) and k.startswith('R-HSA-')}

g_members, cov_rows = {}, []
for name, pid in NAMED_TARGETS.items():
    sub = _desc(pid)
    v2_ids = ([pid] if pid in embedded else []) + sorted(sub & embedded)
    assert len(v2_ids) == FROZEN_SET_SIZES[name], (
        f'member-set lock failed {name}: {len(v2_ids)} != '
        f'{FROZEN_SET_SIZES[name]} (Reactome/embedding drift)')
    g_members[name] = v2_ids
    n_canon = CANON_TOTAL[name]                       # PINNED denominator
    cov_rows.append({
        'pathway_name': name, 'pathway_id': pid,
        'parent_embedded': pid in embedded,
        'n_members_embedded': len(v2_ids),
        'n_descendants_canonical': n_canon,           # dag_metadata frozen
        'coverage_frac': len(v2_ids) / n_canon,       # the value used
        'raw_recompute_total': len(sub),              # cross-check only
        'testable': len(v2_ids) >= MIN_TESTABLE,
    })

coverage = pd.DataFrame(cov_rows)
print('\ngpath2vec arm A · 5-parent per-member coverage '
      '(per pathway, NOT aggregated; coverage_frac pinned to dag_metadata):')
print(coverage.to_string(index=False))
print('\n[transparency] raw_recompute_total = all-level ReactomePathways '
      'recompute; it differs from n_descendants_canonical for some pathways '
      '(all-level vs the frozen AUCell-comparable hierarchy). the table and '
      'all prose use n_descendants_canonical (dag_metadata, '
      'proposal_deviations.md). the raw column is shown only so the gap is '
      'visible, never as the denominator.')

# per-member vectors: L2-normed member matrix (NOT a rollup; each member kept separate)
g_memvec = {}
for name, ids in g_members.items():
    P = np.stack([np.asarray(_emb[i], np.float64) for i in ids])
    P /= (np.linalg.norm(P, axis=1, keepdims=True) + 1e-12)
    g_memvec[name] = P

# per-niche gpath2vec vectors, restricted to R4 test niches (memory)
_r4_ids = set(umaps['R4']['niche_id'])
g_niche = {}
for k, v in _emb.items():
    if isinstance(k, str) and k.startswith('cluster_'):
        nid = k.rsplit('__', 1)[-1]
        if nid in _r4_ids and nid not in g_niche:
            g_niche[nid] = np.asarray(v, np.float64)
del _emb
print(f'\ngpath2vec niche vectors loaded for R4 test set: {len(g_niche):,} / '
      f'{len(_r4_ids):,} R4 test niches')

arm A embeddings.pkl sha256 verified: 5705df87565db27dee1cefdb72583cf63c36d0c1e085daa2defb2421fbd7f8dc
canonical coverage denominator pinned (dag_metadata frozen 395): {'Cell_Cycle': 120, 'ECM_Organization': 20, 'Immune_System': 210, 'Programmed_Cell_Death': 38, 'TGF-beta_Signaling': 7}

gpath2vec arm A · 5-parent per-member coverage (per pathway, NOT aggregated; coverage_frac pinned to dag_metadata):
         pathway_name    pathway_id  parent_embedded  n_members_embedded  n_descendants_canonical  coverage_frac  raw_recompute_total  testable
   TGF-beta_Signaling  R-HSA-170834             True                   2                        7       0.285714                    6     False
        Immune_System  R-HSA-168256            False                  74                      210       0.352381                  218      True
     ECM_Organization R-HSA-1474244            False                  14                       20       0.700000                   19      True
           Cell_Cyc

**reading the coverage table.** `coverage_frac` = `n_members_embedded` / `n_descendants_canonical`, where the denominator is **pinned to the dag_metadata frozen hierarchy** (`dag_metadata.parquet` ∩ `per_node_cca.node_id` = 395 nodes; the AUCell-comparable canonical source per `proposal_deviations.md`): Immune 74/210, ECM 14/20, Cell Cycle 14/120, PCD 7/38, TGF-β 2/7. the cell hard-asserts this against the deviations-log pin and fails loudly on drift, so the headline coverage % cannot silently move (the unpinned-denominator bug class). `raw_recompute_total` is the all-level `ReactomePathwaysRelation.txt` recompute, shown only as a visible transparency cross-check; it is never the denominator. only **Immune (74, 35%)** and **ECM (14, 70%)** clear the n>=5 fair-power bar. **Cell Cycle (14, 12%)** and **PCD (7, 18%)** are sparse subtrees; **TGF-β (2)** is the pinned receptor-complex, structurally underpowered. these three are reported for completeness and are **not** evidence about whether gpath2vec encodes pathway signal. no row is summed into an aggregate.

In [17]:
# §13 descriptive: gpath2vec embedding space laid out by COSINE similarity.
# UMAP under metric=cosine -> niches that are cosine-similar sit together;
# color = Reactome pathway-category to see if cosine clusters track biology.
# v1 (full_cohort_tf_low, pre-F1) precomputed cosine-UMAP coords, reused as-is.
# descriptive; NOT the §13 arm_A/v2 CCA probe; v1 numbers retired.
import datetime as _dt

V1_DIR = BIO / 'gpath2vec_output' / 'full_cohort_tf_low'
v1c = V1_DIR / 'umap_pathway_category_2d_coords.parquet'
assert v1c.is_file(), f'missing v1 cosine-UMAP coords: {v1c}'
_mt = _dt.datetime.fromtimestamp(v1c.stat().st_mtime)
assert _mt.year == 2026 and _mt.month == 4, (
    f'v1 coords mtime {_mt:%Y-%m-%d} not the Apr-2026 v1 artifact - refuse')

_PAR = {'Immune_System': 'Immune System', 'ECM_Organization': 'ECM Organization',
        'Cell_Cycle': 'Cell Cycle', 'Programmed_Cell_Death': 'Programmed Cell Death',
        'TGF-beta_Signaling': 'TGF-beta Signaling'}
_dm = pd.read_parquet(RUNS / 'eval' / 'H3' / 'pathway_cca' / 'dag_full'
                      / 'dag_metadata.parquet', columns=['node_name', 'parent_name'])
name2t5 = dict(zip(_dm['node_name'].astype(str), _dm['parent_name'].map(_PAR)))
_cat2t5 = {'Immune System': 'Immune System',
           'Extracellular matrix organization': 'ECM Organization',
           'Cell Cycle': 'Cell Cycle', 'Programmed Cell Death': 'Programmed Cell Death',
           'Signaling by TGF-beta Receptor Complex': 'TGF-beta Signaling'}

v1 = pd.read_parquet(v1c, columns=['UMAP1', 'UMAP2', 'pathway_category',
                                   'top_pathway', 'patient', 'subarray'])
v1['pathway_category'] = v1['pathway_category'].astype(str)
v1['top_pathway'] = v1['top_pathway'].astype(str)
v1['t5'] = (v1['top_pathway'].map(name2t5)
            .fillna(v1['pathway_category'].map(_cat2t5)).fillna('-'))
print(f'v1 cosine-UMAP: {len(v1):,} niches, '
      f'{v1["pathway_category"].nunique()} pathway-categories, '
      f'{100*(v1["t5"]!="-").mean():.1f}% under a 5-target  '
      f'(coords {_mt:%Y-%m-%d}, cosine metric, pre-F1, NOT arm_A/v2)')

# 5 H3 targets in fixed standard colors (Immune light green); sub-categories
# = shades; non-target = light blue. (matches the agreed scheme.)
_fam = v1.groupby('pathway_category')['t5'].agg(
    lambda s: s.value_counts().idxmax())
BASE5 = {'Immune System': '#74c476', 'ECM Organization': '#ff7f0e',
         'Cell Cycle': '#6baed6', 'Programmed Cell Death': '#d62728',
         'TGF-beta Signaling': '#9467bd'}
# non-target categories each get their own distinct (de-emphasized) color


def _h2r(h):
    return tuple(int(h[i:i + 2], 16) for i in (1, 3, 5))


def _shades(base, n):
    r, g, b = _h2r(base)
    if n <= 1:
        return [base]
    out = []
    for i in range(n):
        f = 0.50 - 1.00 * (i / (n - 1))
        c = ((r + (255 - r) * f, g + (255 - g) * f, b + (255 - b) * f)
             if f >= 0 else (r * (1 + f), g * (1 + f), b * (1 + f)))
        out.append('#%02x%02x%02x' % tuple(int(round(x)) for x in c))
    return out


pc_color, order, _oi = {}, [], 0
for fam in list(BASE5) + ['-']:
    cats = sorted(c for c in _fam.index if _fam[c] == fam)
    if not cats:
        continue
    if fam in BASE5:                       # 5 targets: base hue + shades
        cols = _shades(BASE5[fam], len(cats))
    else:                                  # non-target: a distinct color each
        cols = [PALETTE[(_oi + k) % len(PALETTE)] for k in range(len(cats))]
        _oi += len(cats)
    for cat, col in zip(cats, cols):
        pc_color[cat] = col
        order.append(cat)

d = v1.sample(20000, random_state=SEED) if len(v1) > 20000 else v1
fig = go.Figure()
for cat in order:
    s = d[d['pathway_category'] == cat]
    if s.empty:
        continue
    is_t = _fam[cat] in BASE5
    fig.add_trace(go.Scattergl(
        x=s['UMAP1'], y=s['UMAP2'], mode='markers',
        marker=dict(size=3, color=pc_color[cat], opacity=0.8 if is_t else 0.4),
        name=f'{(_fam[cat] if is_t else "other")} · {cat}'[:40],
        legendgroup=cat,
        text=[f'subarray: {sb}<br>patient: {pt}<br>pathway-category: {cat}'
              f'<br>top_pathway: {tp}<br><b>5-target: {t5}</b>'
              for sb, pt, tp, t5 in zip(s['subarray'], s['patient'],
                                        s['top_pathway'], s['t5'])],
        hovertemplate='%{text}<extra></extra>'))
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(
    title=('v1 gpath2vec embedding space, UMAP under <b>cosine</b> metric '
           '- niches clustered by cosine similarity, colored by Reactome '
           'pathway-category<br><sub>descriptive; NOT the §13 arm_A/v2 CCA '
           'probe; v1 numbers retired (shape only). 5 H3 targets in fixed '
           f'colors (Immune=light green, Cell Cycle=light blue); each non-target category its own de-emphasized color. '
           f'{len(d):,} of {len(v1):,} niches.</sub>'),
    height=720, width=1080, margin=dict(l=20, r=270, t=110, b=20),
    legend=dict(itemsizing='constant', x=1.02, y=1.0, font=dict(size=8)))
fig.show()

v1 cosine-UMAP: 286,233 niches, 19 pathway-categories, 35.0% under a 5-target  (coords 2026-04-21, cosine metric, pre-F1, NOT arm_A/v2)


one claim: in the gpath2vec embedding's own space (UMAP under the cosine metric), cosine-similar niches cluster together; coloring by Reactome pathway-category shows whether those cosine clusters track pathway biology. this is descriptive (v1, not the §13 arm_A/v2 CCA probe); the quantitative per-member CCA test is below.

In [18]:
H3G = RUNS / 'eval' / 'H3' / 'pathway_cca_gpath2vec_vs_aucell_tflow'
pp = H3G / 'per_pathway_cca.parquet'

if not pp.exists():
    print('PENDING: per_pathway_cca.parquet not written yet.')
    print(f'  expected at: {pp.relative_to(ROOT)}')
    print('  written once after the full grid finishes; rerun this cell '
          'when runs/.../run.log shows the grid complete. nothing '
          'fabricated in the interim.')
    g_cca = None
else:
    cca = pd.read_parquet(pp)
    # gpath2vec arm ONLY (aucell_lowmatched = separate head-to-head, not here)
    g_cca = cca[cca['method'] == 'gpath2vec'].copy()
    g_cca = g_cca.merge(
        coverage[['pathway_name', 'n_members_embedded', 'coverage_frac']],
        on='pathway_name', how='left')
    g_cca['underpowered'] = g_cca['shared_size'] < MIN_TESTABLE

    def _row_color(r):
        if r['underpowered']:
            return '#ececec'                      # underpowered (TGF-b, n<5)
        if bool(r['sig_A_05']):
            return '#d8f0d8'                      # BH-FDR significant
        return '#f7dcdc'                          # not significant

    def _fmt(d):
        f = pd.DataFrame()
        if 'run_id' in d:
            f['run'] = d['run_id'].values
        f['pathway'] = d['pathway_name'].str.replace('_', ' ').values
        f['n'] = d['shared_size'].astype(int).values
        f['cov'] = d['coverage_frac'].map('{:.2f}'.format).values
        f['n_test'] = d['n_test'].astype(int).values
        f['r'] = d['top_corr_test'].map('{:.3f}'.format).values
        f['z_A'] = d['z_A'].map('{:+.2f}'.format).values
        f['p'] = d['p_two_A'].map('{:.2e}'.format).values
        f['BH-FDR'] = d['fdr_A'].map(
            lambda x: '-' if pd.isna(x) else f'{x:.3f}').values
        f['sig'] = d['sig_A_05'].map(lambda b: 'YES' if b else 'no').values
        return f

    def _table(d, title):
        rc = d.apply(_row_color, axis=1).tolist()
        f = _fmt(d)
        fig = go.Figure(go.Table(
            header=dict(values=[f'<b>{c}</b>' for c in f.columns],
                        fill_color='#3b3b3b',
                        font=dict(color='white', size=11),
                        align='left', height=26),
            cells=dict(values=[f[c].tolist() for c in f.columns],
                       fill_color=[rc] * len(f.columns),
                       font=dict(size=11), align='left', height=22)))
        fig.update_layout(title=dict(text=title, font=dict(size=13)),
                          height=70 + 22 * len(f), width=900,
                          margin=dict(l=10, r=10, t=48, b=8))
        return fig

    head = g_cca[(g_cca['run_id'] == 'R4') & (g_cca['view'] == 'z_he')] \
        .sort_values('pathway_name')
    _table(head, 'H3 headline | gpath2vec | R4 | z_he (clean) | per pathway, '
                 'no aggregate  (green = BH-FDR sig, red = not, '
                 'grey = underpowered n&lt;5)').show()

    grid = g_cca[g_cca['view'] == 'z_he'].sort_values(
        ['pathway_name', 'run_id'])
    _table(grid, 'all runs | z_he (clean) | per pathway | classical '
                 'B1/B2/B3 vs contrastive R1/R2/R3/R4/R6 | read per pathway, '
                 'no aggregate').show()

### interpretation (final, computed)

source of record = the cell directly above (`per_pathway_cca.parquet`, `method=='gpath2vec'`, R4 z_he). numbers below are read from that computed table, not a preview. per pathway, `z_he` only, no aggregate. R4 z_he is the headline (`view_clean=True`, the non-circular shared aligned space the proposal targets).

**R4 z_he, gpath2vec, BH-FDR over the testable family:**

- **ECM** (n=14, 70% coverage): z=7.25, BH-FDR=0.006 -> **significant**. the only well-covered target; proposal H3 supported here most strongly.
- **Immune** (n=74, 35% coverage): z=6.52, BH-FDR=0.006 -> **significant**. proposal H3 supported.
- **Cell Cycle** (n=14, 12% coverage): z=6.05, BH-FDR=0.006 -> **significant**, but over a thin 12% subtree: report as signal on **the embedded low-level subset**, not a whole-Cell-Cycle claim.
- **PCD** (n=7, 18% coverage): z=2.61, BH-FDR=**0.050** (0.050185) -> **does not survive correction** at the pre-registered alpha=0.05. reported as a **marginal / suggestive** association, not a confirmed hit; doubly weak (boundary FDR and 18% subtree coverage). exact value shown, not hidden.
- **TGF-β** (n=2): z=2.78, BH-FDR=NaN -> **untestable by construction** (n=2 < 5, the only `underpowered=True` row; excluded from the FDR family). carries no interpretation.

**proposal H3, honest read**: on the R4 shared latent, gpath2vec pathway embeddings show BH-FDR-significant CCA correlation for **Immune, ECM, and Cell Cycle (z 6.0-7.2)**; **PCD is marginal and does not survive correction**; **TGF-β is untestable (n=2)**. ECM is the only well-covered target and the only pathway where gpath2vec exceeds the member-matched AUCell baseline. a partial-positive, proposal-anticipated result ("negative or partial results establish the noise floor"), not a clean win and not a null.

**falsifier contrast** (gpath2vec, z_he, 4 testable pathways, BH-FDR-significant count): R4 3/4, classical B1/B2/B3 3/4 each, R6 1/4 (Immune only), R1/R2/R3 0/4. gpath2vec pathway signal transfers in R4 and the classical aligned spaces and fails in the late-fusion contrastive runs, matching the established "R4 is the H3 performer, late-fusion fails H3" finding. classical baselines match R4 here, so the alignment-specific reading is that cross-attention does not lose the signal, not that it uniquely creates it.

**limit, stated**: this section is gpath2vec-only by design. whether gpath2vec *exceeds* AUCell is the separate controlled head-to-head (`aucell_lowmatched` rows, same parquet, not shown here): gpath2vec beats member-matched AUCell only on ECM, loses on Immune / Cell Cycle / PCD. AUCell is the stronger per-niche representation overall; gpath2vec's edge is narrow and coverage-dependent. residual confound: gpath2vec EA additionally restricted pathway genes to the TF-low universe (harness docstring `RECOMPUTE_TF`). the gpath2vec cosine-similarity UMAP above is descriptive of v1 embedding structure; the low-level coverage dilution still motivates the planned all-level no-TF re-embed.

## 14 · summary

what these UMAPs visualize:

| section | result it visualizes |
|---|---|
| §3 (mc_megacluster) | H2 part A-MC niche-level rescue: R1 and R6 show clear MC clusters (KMeans ARI 0.23-0.25); R4's tight manifold compresses the same biology into linearly-separable but non-globular shapes (linear probe parity confirms biology is there, K-means geometry bias makes it look weak) |
| §4 (9-archetype, proposal-literal H2 part A) | the label the proposal H2 names. **patient-level pseudobulk** (every niche in a patient inherits one archetype): ARI on this label is mechanically a 9-class patient classification at niche resolution. shown across all 8 runs for transparency. |
| §5 (construct-validity, archetype vs patient_id) | **visual punchline**: B1 (CCA) collapses to patient identity - its archetype panel and patient_id panel are structurally identical. R4 mixes patients - the two overlays diverge. NMI(archetype, patient_id) quantifies the collapse. answers "is the H2 part A evaluation question good?" - no, not as the proposal literally writes it. |
| §6 (patient_id) | the patient-leakage axis itself: B1 CCA shows tight per-patient clusters (anti-helpful, patient z 122.13); R4 mixes patients best (patient z 22.97, lowest); contrastive runs (R1/R2/R3) sit between |
| §7 (17 tissue compartments, proposal H2 part B) | per-spot histopathologist-annotated compartments (Tumor, High/Low TIL stroma, TLS, Necrosis, ...) - these DO vary at niche resolution, so they support a real spatial-coherence test (13/38 subarrays covered, 12 classes populated). this is the proposal H2 part B per-compartment readability label. |
| §8 (z_he / z_st / z_mean triptych for R4 and B1) | proposal H2 across "three embedding spaces": post-projection H&E view, post-projection ST view, joint mean view. shows how the same niches and same biology labels distribute across the three views of the shared latent. un-aligned modality view (raw Virchow2, raw Novae) deferred to follow-up. |
| §9 (5-parent AUCell on R4) | H3 part A pathway gradients: each named Reactome parent fires in spatial patterns across R4's manifold; all 5 panels look similar within R4 (specificity off-diag 0.819 at 5-parent test) |
| §10 (DAG canonical directions on R4) | Reactome DAG (pathway hierarchy) finding: R4 distinguishes biologically distinct sub-pathways (BTLA, IL/SHC, IFN, integrin, ECM proteoglycans) at DAG resolution; aliasing was a 5-parent coarse-aggregation artifact, NOT an R4 encoding property |
| §11 (sig DAG-tests per run bar chart) | H3 cross-patient: 1185 tests/run (395 nodes × 3 views). R4 78.8%, classical B2 74.2%/B1 70.2%/B3 64.4% competitive at DAG resolution; late-fusion contrastive R6 36.9%/R1 27.1%/R2 23.0%/R3 18.4% |
| §12 (run × parent heatmap) | parent-resolved R4 dominance: R4 wins or ties on every parent; ECM is universally easiest, Cell Cycle hardest |
| §13 (gpath2vec pathway-CCA, arm A) | H3 proposal-literal test: gpath2vec per-member multivariate CCA vs shared latent (no rollup), headline R4 z_he, reported per pathway with coverage, no aggregate. populates from `per_pathway_cca.parquet` (`method=='gpath2vec'`); the proposal-literal arm, distinct from the §9-§12 AUCell substitution. |

the integrated read: cross-attention bridge (R4) wins H1 retrieval and H3 cross-patient pathway generalization. late-fusion contrastive (R1, R6) wins H2 niche-level biology preservation. classical baselines lose H1 but are competitive at DAG resolution (commit-3.6: classical catch up at fine pathway resolution). proposal H2 part A as literally written is mechanically a patient classification (construct-validity issue documented in §4-§5); the niche-level rescue is `mc_megacluster` in §3 with the K-means geometry-bias caveat (this is possibly leaning towards an annotation quality issue).